In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import *
import requests
import random
from delta.tables import DeltaTable
from datetime import datetime

#Campi che "potrebbero cambiare" in una birreria reale

FAKE_PHONE_PREFIXES = ["+1-500", "+1-800", "+1-900", "+44-20", "+39-081"]
FAKE_NAME_SUFFIXES = ["Brewing Co.", "Craft Beer", "Ale House", "Brewery", "Taproom", "Beer Works"]
FAKE_STREET_TYPES = ["Main St", "Oak Ave", "Elm St", "Brewery Ln", "Craft Blvd", "Hop Rd"]

def genera_telefono_casuale():
    prefix = random.choice(FAKE_PHONE_PREFIXES)
    number = random.randint(1000000, 9999999)
    return f"{prefix}-{number}"

def genera_indirizzo_casuale():
    n = random.randint(1, 999)
    street = random.choice(FAKE_STREET_TYPES)
    return f"{n} {street}"

def genera_strada_casuale(original_name):
    base = original_name.split(" ")[0] #prende la prima parola
    suffix = random.choice(FAKE_NAME_SUFFIXES)
    return f"{base} {suffix}"

In [0]:
#Quante birrerie modificare ad ogni run
NUM_BREWERIES_UPDATE = random.randint(3, 10)
print(f"N_MODIFICHE pianificate: {NUM_BREWERIES_UPDATE}")
silver = spark.read.table("notebook_breweries.silver_staging_breweries")

#Prendere N record casuali distinti

sample_ids = (silver
                .select("id")
                .distinct()
                .orderBy(F.rand())
                .limit(NUM_BREWERIES_UPDATE)
                .collect()    
            )
print(f"Sample ids raccolti: {len(sample_ids)}")

rows_to_insert = []

for row in sample_ids:
    brewery_id = row["id"]

    #Prendere ultima versione di quella birreria
    original = (
        silver
            .filter(F.col("id") == brewery_id)
            .orderBy(F.col("ingestion_ts").desc())
            .limit(1)
            .collect()
    )

    original = original[0]

    #Scegliere casualmente quale campo modificare 
    campo = random.choice(["phone", "street", "name"])

    new_row = original.asDict()
    new_row["ingestion_ts"] = datetime.now() #nuovo timestamp -> SCD2 lo vede come un update

    if campo == "phone":
        new_row["phone"] = genera_telefono_casuale()

    elif campo == "street":
        new_row["street"] = genera_indirizzo_casuale()

    elif campo == "name":
        new_row["name"] = genera_strada_casuale(original["name"])

    print(f" [{campo}] {original[campo]} → {new_row[campo]} ({brewery_id})")

    rows_to_insert.append(new_row)

print(f"Totale rows_to_insert: {len(rows_to_insert)}")

In [0]:
%skip
if len(rows_to_insert) > 0:
    schema = silver.schema
    modified_df = spark.createDataFrame(rows_to_insert, schema=schema)
    modified_df.write.format("delta").mode("append").saveAsTable("notebook_breweries.silver_breweries")
    print(f"Scritti {len(rows_to_insert)} record")
    print(f"Totale silver dopo append: {spark.read.table('notebook_breweries.silver_breweries').count()}")
else:
    print("rows_to_insert è vuoto — nessun record scritto")


In [0]:

#Inserire i record modificati nella silver come nuovi arrivi
schema = silver.schema

modified_df = spark.createDataFrame(rows_to_insert, schema = schema)

(modified_df
    .write
    .format("delta")
    .mode("append")
    .saveAsTable("notebook_breweries.silver_staging_breweries")
)